# Check Regular Projection

Notebook simple pour verifier rapidement un chunk projete sur le maillage regulier.

Checks inclus :
- compatibilite entre le nombre de noeuds du maillage et les tenseurs `(x, y)`;
- continuite des `ts` dans le chunk;
- statistiques rapides sur `h`, `u`, `v`, `delta_h`, `delta_u`, `delta_v`;
- visualisation sur le maillage regulier.


In [ ]:
from pathlib import Path
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / 'python' / 'create_dgl_dataset.py').exists():
            return path
    raise RuntimeError('Cannot find gnn_modulus_test project root.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.create_dgl_dataset import add_mesh_info

os.chdir(PROJECT_ROOT)
PROJECT_ROOT


## Parametres a remplir

In [ ]:
REGULAR_MESH_SLF = ''
PROJECTED_DYNAMIC_PKL = ''
PROJECTED_BASE_BIN = ''  # optionnel
SAMPLE_INDEX = 0
WET_THRESHOLD = 0.05
FIGSIZE = (16, 4)


In [ ]:
def require_path(path_like: str, name: str) -> Path:
    if not path_like:
        raise ValueError(f'{name} is empty.')
    path = Path(path_like).expanduser()
    if not path.exists():
        raise FileNotFoundError(f'{name} does not exist: {path}')
    return path


def unpack_sample(sample):
    if isinstance(sample, dict):
        return sample['x'], sample['y'], sample.get('ts', None)
    if len(sample) == 2:
        x, y = sample
        return x, y, None
    if len(sample) == 3:
        x, y, ts = sample
        return x, y, ts
    raise ValueError('Unsupported sample format')


def var_stats(name: str, values: np.ndarray) -> dict:
    return {
        'var': name,
        'min': float(np.min(values)),
        'max': float(np.max(values)),
        'mean': float(np.mean(values)),
        'std': float(np.std(values)),
        'zero_ratio': float(np.mean(values == 0.0)),
        'finite_ratio': float(np.mean(np.isfinite(values))),
    }


In [ ]:
mesh_path = require_path(REGULAR_MESH_SLF, 'REGULAR_MESH_SLF')
dynamic_path = require_path(PROJECTED_DYNAMIC_PKL, 'PROJECTED_DYNAMIC_PKL')

mesh = TelemacFile(str(mesh_path))
X, triangles = add_mesh_info(mesh)
triangulation = mtri.Triangulation(X[:, 0], X[:, 1], triangles)

with dynamic_path.open('rb') as file_obj:
    raw_samples = pickle.load(file_obj)

samples = [unpack_sample(sample) for sample in raw_samples]
if not samples:
    raise ValueError('The projected chunk is empty.')

x0, y0, ts0 = samples[SAMPLE_INDEX]

assert x0.shape == (len(X), 3), f'Expected x shape {(len(X), 3)}, got {x0.shape}'
assert y0.shape == (len(X), 3), f'Expected y shape {(len(X), 3)}, got {y0.shape}'

ts_values = [ts for _, _, ts in samples if ts is not None]
ts_diffs = np.diff(ts_values) if len(ts_values) > 1 else np.array([])

print('mesh nodes:', len(X))
print('mesh triangles:', len(triangles))
print('chunk length:', len(samples))
print('sample index:', SAMPLE_INDEX)
print('sample ts:', ts0)
print('unique ts diffs:', np.unique(ts_diffs).tolist() if len(ts_diffs) else [])


In [ ]:
stats = pd.DataFrame([
    var_stats('x_h', x0[:, 0]),
    var_stats('x_u', x0[:, 1]),
    var_stats('x_v', x0[:, 2]),
    var_stats('y_dh', y0[:, 0]),
    var_stats('y_du', y0[:, 1]),
    var_stats('y_dv', y0[:, 2]),
])

stats


In [ ]:
speed0 = np.linalg.norm(x0[:, 1:3], axis=1)
wet_mask = x0[:, 0] > WET_THRESHOLD

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)

plots = [
    ('Projected h', x0[:, 0], 'viridis'),
    ('Projected speed', speed0, 'magma'),
    ('Projected delta_h', y0[:, 0], 'coolwarm'),
]

for ax, (title, values, cmap) in zip(axes, plots):
    artist = ax.tripcolor(triangulation, values, shading='flat', cmap=cmap)
    ax.set_title(title)
    ax.set_aspect('equal')
    fig.colorbar(artist, ax=ax, shrink=0.8)

plt.show()

print('wet fraction (> threshold):', float(np.mean(wet_mask)))


In [ ]:
if PROJECTED_BASE_BIN:
    import dgl

    base_path = require_path(PROJECTED_BASE_BIN, 'PROJECTED_BASE_BIN')
    graphs, _ = dgl.load_graphs(str(base_path))
    base_graph = graphs[0]

    print('base graph nodes:', base_graph.num_nodes())
    print('base graph edges:', base_graph.num_edges())
    print('static shape:', tuple(base_graph.ndata['static'].shape))

    assert base_graph.num_nodes() == len(X), 'Mismatch between base graph and regular mesh'
    assert base_graph.ndata['static'].shape[1] == 6, 'Expected 6 static features'


## Lecture rapide

Si ce notebook passe sans assertion et que les cartes n'ont pas de zones aberrantes, tu as deja valide le coeur de la projection.

Points a surveiller :
- `unique ts diffs` doit en general etre `[1]`;
- `finite_ratio` doit etre a `1.0` pour toutes les variables;
- `zero_ratio` ne doit pas exploser de maniere inattendue d'un chunk a l'autre;
- le `base.bin` doit avoir exactement le meme nombre de noeuds que le maillage regulier.
